In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no-SP","SP"])
fig, ax = plt.subplots(figsize=(4,4))
disp.plot(ax=ax)
plt.show()

In [ ]:
import pandas as pd
coefs = clf.named_steps["lr"].coef_[0]
imp = pd.DataFrame({"feature": feat_names, "coef": coefs, "abs": np.abs(coefs)}).sort_values("abs", ascending=False)
imp.head(20)

In [ ]:
import pandas as pd
import numpy as np

# Dohvati koeficijente
coefs = clf.named_steps["lr"].coef_[0]

importance_df = pd.DataFrame({
    "feature": feat_names,
    "coef": coefs,
    "abs_coef": np.abs(coefs)
}).sort_values("abs_coef", ascending=False)

importance_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top_k = 15
top_feats = importance_df.head(top_k)

plt.figure(figsize=(8,6))
plt.barh(top_feats["feature"], top_feats["coef"])
plt.gca().invert_yaxis()
plt.title("Top Logistic Regression Coefficients")
plt.xlabel("Coefficient value")
plt.show()

In [ ]:
importance_df["odds_ratio"] = np.exp(importance_df["coef"])
importance_df.head(10)

In [ ]:
import seaborn as sns

feat_index = feat_names.index("sp_hydro_frac_25")

sns.kdeplot(X_train[y_train==1][:,feat_index], label="SP")
sns.kdeplot(X_train[y_train==0][:,feat_index], label="No SP")
plt.legend()
plt.title("Distribution of N-terminal Hydrophobic Fraction")
plt.show()

In [ ]:
from scipy.stats import ttest_ind

sp_vals = X_train[y_train==1][:, feat_names.index("sp_hydro_frac_25")]
nsp_vals = X_train[y_train==0][:, feat_names.index("sp_hydro_frac_25")]

print(ttest_ind(sp_vals, nsp_vals))

In [ ]:
import numpy as np

sp_vals = X_train[y_train==1][:, feat_names.index("sp_hydro_frac_25")]
nsp_vals = X_train[y_train==0][:, feat_names.index("sp_hydro_frac_25")]

mean_diff = sp_vals.mean() - nsp_vals.mean()
pooled_std = np.sqrt((sp_vals.var() + nsp_vals.var()) / 2)

cohens_d = mean_diff / pooled_std
cohens_d

In [ ]:
import numpy as np
import pandas as pd

# Grupe po imenima (prema feat_names koje smo ranije definirali)
aac_idx = [i for i, n in enumerate(feat_names) if n.startswith("aac_")]

nterm_idx = [feat_names.index(n) for n in [
    "kr_cluster_50", "sp_hydro_frac_25", "sp_kd_mean_25", "sp_motif_AxA"
] if n in feat_names]

# "Global physchem" = sve što nije AAC i nije N-terminal proxy
all_idx = list(range(len(feat_names)))
global_idx = [i for i in all_idx if (i not in aac_idx) and (i not in nterm_idx)]

print("AAC dims:", len(aac_idx))
print("Global physchem dims:", len(global_idx))
print("N-term proxy dims:", len(nterm_idx))

# sanity check: pokrivamo sve dimenzije
covered = set(aac_idx) | set(global_idx) | set(nterm_idx)
print("Total dims:", len(all_idx), "Covered:", len(covered), "Missing:", set(all_idx) - covered)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

def fit_eval_lr(Xtr, ytr, Xte, yte, cols, name):
    Xtr_s = Xtr[:, cols]
    Xte_s = Xte[:, cols]

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000))
    ])
    clf.fit(Xtr_s, ytr)

    pred = clf.predict(Xte_s)
    proba = clf.predict_proba(Xte_s)[:, 1]

    return {
        "setting": name,
        "dims": len(cols),
        "accuracy": accuracy_score(yte, pred),
        "macro_f1": f1_score(yte, pred, average="macro"),
        "roc_auc": roc_auc_score(yte, proba),
        "pr_auc": average_precision_score(yte, proba)
    }

In [ ]:
results = []

# 1) Full (sve)
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, all_idx, "FULL (AAC + global + N-term)"))

# 2) Bez N-terminal proxy (najvažniji test)
cols_no_nterm = [i for i in all_idx if i not in nterm_idx]
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, cols_no_nterm, "ABLATION: FULL - N-term"))

# 3) Samo N-terminal proxy (koliko je “jak” sam)
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, nterm_idx, "ONLY: N-term proxy"))

# 4) Samo AAC
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, aac_idx, "ONLY: AAC"))

# 5) Samo global physchem (bez AAC, bez N-term)
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, global_idx, "ONLY: global physchem"))

# 6) AAC + N-term (bez global)
cols_aac_nterm = sorted(list(set(aac_idx) | set(nterm_idx)))
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, cols_aac_nterm, "AAC + N-term (no global)"))

# 7) AAC + global (bez N-term)
cols_aac_global = sorted(list(set(aac_idx) | set(global_idx)))
results.append(fit_eval_lr(X_train, y_train, X_test, y_test, cols_aac_global, "AAC + global (no N-term)"))

res_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
res_df

In [ ]:
res_df_round = res_df.copy()
for c in ["accuracy", "macro_f1", "roc_auc", "pr_auc"]:
    res_df_round[c] = res_df_round[c].map(lambda x: round(float(x), 4))

res_df_round

In [ ]:
import matplotlib.pyplot as plt

# ROC-AUC
plt.figure(figsize=(9,4))
plt.bar(res_df_round["setting"], res_df_round["roc_auc"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("ROC-AUC")
plt.title("Ablation Study (LogReg) — ROC-AUC")
plt.tight_layout()
plt.show()

# PR-AUC
plt.figure(figsize=(9,4))
plt.bar(res_df_round["setting"], res_df_round["pr_auc"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("PR-AUC")
plt.title("Ablation Study (LogReg) — PR-AUC")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

feat_index = feat_names.index("sp_hydro_frac_25")

plt.figure(figsize=(6,4), dpi=300)

sns.kdeplot(X_train[y_train==1][:,feat_index],
            label="Signal peptide",
            linewidth=2)

sns.kdeplot(X_train[y_train==0][:,feat_index],
            label="Non-signal",
            linewidth=2)

plt.xlabel("N-terminal hydrophobic fraction (first 25 residues)")
plt.ylabel("Density")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("Figure1_hydrophobic_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(6,5), dpi=300)

top_feats = importance_df.head(12)

plt.barh(top_feats["feature"], top_feats["coef"])
plt.gca().invert_yaxis()

plt.xlabel("Logistic regression coefficient")
plt.title("Top predictive features")
plt.tight_layout()
plt.savefig("Figure2_feature_importance.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7,4), dpi=300)

plt.bar(res_df_round["setting"], res_df_round["roc_auc"])

plt.xticks(rotation=60, ha="right")
plt.ylabel("ROC-AUC")
plt.tight_layout()
plt.savefig("Figure3_ablation.png", dpi=300)
plt.show()